<a href="https://colab.research.google.com/github/ran4erep/Torrent-Loader/blob/main/Torrent_loader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title 📥 Torrent Downloader { display-mode: "form" }
# @markdown <-- Нажмите эту кнопку для старта интерфейса
import os
import re
import subprocess
import shutil
import glob
import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import drive

# 1. Ensure aria2 is installed quietly
status = subprocess.run(['which', 'aria2c'], capture_output=True)
if status.returncode != 0:
    subprocess.run(['apt-get', '-y', 'install', 'aria2'], capture_output=True)

clear_output()

# --- Custom CSS for word wrap ---
display(widgets.HTML("<style>.widget-checkbox .widget-label { white-space: normal !important; word-break: break-all !important; line-height: 1.2 !important; }</style>"))

# --- UI Components ---
style = {'description_width': 'initial'}
header = widgets.HTML("<h2>🚀 Torrent Downloader</h2>")

input_url = widgets.Text(description="Magnet/URL:", placeholder="Вставьте ссылку", layout=widgets.Layout(width='98%'), style=style)
file_upload = widgets.FileUpload(accept='.torrent', description="Файл .torrent", layout=widgets.Layout(width='98%'))
use_drive_cb = widgets.Checkbox(value=False, description='Сохранять на Google Drive', style=style)

btn_load = widgets.Button(description="Загрузить структуру файлов", button_style='info', icon='sitemap', layout=widgets.Layout(width='98%'))
files_box = widgets.VBox([], layout=widgets.Layout(max_height='400px', overflow='auto', border='1px solid #ddd', padding='8px', margin='5px 0'))

input_box = widgets.VBox([widgets.HTML("<b>1. Источник:</b>"), input_url, file_upload, btn_load, files_box, use_drive_cb], layout=widgets.Layout(border='1px solid #ddd', padding='10px', margin='5px'))

btn_start = widgets.Button(description="Начать скачивание", button_style='primary', icon='download', layout=widgets.Layout(width='98%'))
control_box = widgets.HBox([btn_start], layout=widgets.Layout(margin='10px 0'))

prog_bar = widgets.FloatProgress(value=0.0, min=0.0, max=100.0, description='Прогресс:', bar_style='info', layout=widgets.Layout(width='98%'), style=style)
status_label = widgets.HTML("<b>Статус:</b> Готов к работе")
info_label = widgets.HTML("")
progress_box = widgets.VBox([widgets.HTML("<b>2. Состояние:</b>"), status_label, prog_bar, info_label], layout=widgets.Layout(border='1px solid #ddd', padding='10px', margin='5px'))

app_ui = widgets.VBox([header, input_box, control_box, progress_box])

# --- Logic ---
current_proc_container = {'process': None, 'torrent_path': None, 'checkboxes': {}, 'file_paths': {}}

def update_ui(status_text, progress=0, info=""):
    status_label.value = f"<b>Статус:</b> {status_text}"
    prog_bar.value = progress
    info_label.value = info

def build_tree_widgets(current_dict, file_checkboxes):
    nodes = []
    all_cbs_in_branch = []

    for node_name, content in current_dict.items():
        if isinstance(content, dict):
            folder_cb = widgets.Checkbox(value=True, indent=False, layout=widgets.Layout(width='35px', margin='0'))
            folder_label = widgets.HTML(f"<div style='word-wrap: break-word; white-space: normal; padding-top: 4px;'>📁 <b>{node_name}</b></div>")
            folder_hbox = widgets.HBox([folder_cb, folder_label], layout=widgets.Layout(align_items='flex-start'))

            children_nodes, children_cbs = build_tree_widgets(content, file_checkboxes)
            folder_vbox = widgets.VBox(children_nodes, layout=widgets.Layout(margin='0 0 0 20px'))

            def make_toggle(cbs):
                def toggle(change):
                    for c in cbs:
                        c.value = change['new']
                return toggle

            folder_cb.observe(make_toggle(children_cbs), names='value')

            acc = widgets.Accordion(children=[folder_vbox], selected_index=0)
            acc.set_title(0, "Раскрыть")

            nodes.append(widgets.VBox([folder_hbox, acc]))
            all_cbs_in_branch.append(folder_cb)
            all_cbs_in_branch.extend(children_cbs)

    for node_name, content in current_dict.items():
        if isinstance(content, str):
            cb = widgets.Checkbox(value=True, indent=False, layout=widgets.Layout(width='35px', margin='0'))
            file_label = widgets.HTML(f"<div style='word-wrap: break-word; white-space: normal; padding-top: 4px;'>📄 {node_name}</div>")
            file_hbox = widgets.HBox([cb, file_label], layout=widgets.Layout(align_items='flex-start'))

            file_checkboxes[content] = cb
            nodes.append(file_hbox)
            all_cbs_in_branch.append(cb)

    return nodes, all_cbs_in_branch

def load_files_list(b):
    update_ui("Получение метаданных...", 0)
    files_box.children = [widgets.HTML("<i>Анализ торрента...</i>")]
    current_proc_container['checkboxes'].clear()
    current_proc_container['file_paths'].clear()

    target = ""
    if file_upload.value:
        fname = list(file_upload.value.keys())[0]
        target = os.path.join("/content", fname)
        with open(target, "wb") as f:
            f.write(file_upload.value[fname]['content'])
    elif input_url.value:
        target = input_url.value
        if target.startswith('magnet:'):
            meta_dir = '/content/torrent_meta'
            if os.path.exists(meta_dir): shutil.rmtree(meta_dir)
            os.makedirs(meta_dir, exist_ok=True)

            subprocess.run(['aria2c', '--bt-metadata-only=true', '--bt-save-metadata=true', '-d', meta_dir, target], capture_output=True)
            t_files = glob.glob(f'{meta_dir}/*.torrent')

            if t_files:
                target = t_files[0]
            else:
                files_box.children = [widgets.HTML("<font color='red'>Не удалось получить метаданные.</font>")]
                update_ui("Ошибка загрузки метаданных", 0)
                return
    else:
        files_box.children = [widgets.HTML("<font color='red'>Укажите источник!</font>")]
        return

    current_proc_container['torrent_path'] = target
    res = subprocess.run(['aria2c', '--show-files', target], capture_output=True, text=True)

    path_tree = {}
    for line in res.stdout.split('\n'):
        match = re.match(r'^\s*(\d+)\|(.+)', line)
        if match:
            idx, full_path = match.groups()
            full_path = full_path.strip()
            current_proc_container['file_paths'][idx] = full_path

            parts = full_path.split('/')
            current_level = path_tree
            for part in parts[:-1]:
                if part not in current_level:
                    current_level[part] = {}
                current_level = current_level[part]
            current_level[parts[-1]] = idx

    file_checkboxes = {}
    ui_nodes, _ = build_tree_widgets(path_tree, file_checkboxes)
    current_proc_container['checkboxes'] = file_checkboxes

    if ui_nodes:
        files_box.children = ui_nodes
        update_ui("Дерево файлов построено. Выберите нужные элементы.", 0)
    else:
        files_box.children = [widgets.HTML("<font color='red'>Не удалось распарсить файлы.</font>")]

def start_download(b):
    if not current_proc_container.get('torrent_path'):
        update_ui("<font color='red'>Сначала загрузите структуру файлов!</font>")
        return

    use_drive = use_drive_cb.value
    local_path = "/content/temp_downloads" if use_drive else "/content/downloads"
    drive_path = "/content/drive/MyDrive/TorrentDownloads"

    if use_drive and not os.path.exists('/content/drive'):
        update_ui("Подключение Google Drive...", 0)
        drive.mount('/content/drive')
        os.makedirs(drive_path, exist_ok=True)

    if os.path.exists(local_path): shutil.rmtree(local_path)

    os.makedirs(local_path, exist_ok=True)

    selected_idxs = [idx for idx, cb in current_proc_container['checkboxes'].items() if cb.value]
    if not selected_idxs:
        update_ui("<font color='red'>Не выбрано ни одного файла!</font>")
        return

    select_file_arg = ','.join(selected_idxs)
    update_ui("Инициализация скачивания...", 0)

    cmd = [
        'aria2c',
        '--dir=' + local_path,
        '--seed-time=0',
        '--summary-interval=1',
        '--bt-max-peers=100',
        '--max-connection-per-server=16'
    ]

    if len(selected_idxs) != len(current_proc_container['checkboxes']):
        cmd.append(f'--select-file={select_file_arg}')

    cmd.append(current_proc_container['torrent_path'])

    current_proc_container['process'] = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

    for line in current_proc_container['process'].stdout:
        if '%' in line and 'DL:' in line:
            try:
                p_val_match = re.search(r'\((\d+)%\)', line)
                p_val = int(p_val_match.group(1)) if p_val_match else prog_bar.value

                size_match = re.search(r'\s([0-9A-Za-z.]+/[0-9A-Za-z.]+)\(', line)
                size_info = size_match.group(1).replace('iB', 'B') if size_match else "?/?"

                speed_match = re.search(r'DL:([^\s]+)', line)
                speed = speed_match.group(1).replace('iB', 'B') if speed_match else "0B"

                eta_match = re.search(r'ETA:([^\s\]]+)', line)
                eta = eta_match.group(1) if eta_match else "..."

                peers_match = re.search(r'CN:(\d+)', line)
                peers = peers_match.group(1) if peers_match else "0"

                seeds_match = re.search(r'SD:(\d+)', line)
                seeds = seeds_match.group(1) if seeds_match else "0"

                info_text = f"Скачано: <b>{size_info}</b> | Скорость: <b>{speed}</b> | Осталось: <b>{eta}</b> | Пиры/Сиды: <b>{peers}/{seeds}</b>"
                update_ui("Скачивание...", p_val, info_text)
            except:
                pass

    exit_code = current_proc_container['process'].wait()

    if exit_code == 0:
        if len(selected_idxs) != len(current_proc_container['checkboxes']):
            update_ui("Очистка ненужных файлов...", 100)
            for idx, fpath in current_proc_container['file_paths'].items():
                if idx not in selected_idxs:
                    full_del_path = os.path.join(local_path, fpath)
                    if os.path.exists(full_del_path):
                        if os.path.isfile(full_del_path):
                            os.remove(full_del_path)

            for root, dirs, files in os.walk(local_path, topdown=False):
                for name in dirs:
                    try:
                        os.rmdir(os.path.join(root, name))
                    except OSError:
                        pass

        if use_drive:
            update_ui("Перенос на Google Drive...", 100, "Пожалуйста, не закрывайте вкладку...")
            # Добавлен параметр --exclude='*.aria2' для предотвращения копирования этих файлов
            subprocess.run(f"rsync -avP --exclude='*.aria2' '{local_path}/' '{drive_path}/'", shell=True)
            shutil.rmtree(local_path)
            update_ui("<font color='green'>Готово!</font>", 100, f"Файлы сохранены в Drive/TorrentDownloads")
        else:
            # Очистка .aria2 файлов для локальной папки
            for aria_file in glob.glob(f"{local_path}/**/*.aria2", recursive=True):
                try:
                    os.remove(aria_file)
                except OSError:
                    pass
            update_ui("<font color='green'>Готово!</font>", 100, f"Файлы сохранены в /content/downloads")
    else:
        update_ui("<font color='red'>Скачивание завершилось с ошибкой.</font>")

btn_load.on_click(load_files_list)
btn_start.on_click(start_download)
display(app_ui)